<a href="https://colab.research.google.com/github/ID26S422/The-Feature-Matching-Project/blob/main/SuperGlue.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/magicleap/SuperGluePretrainedNetwork.git

In [ ]:
!ls

In [ ]:
!ls SuperGluePretrainedNetwork

In [ ]:
!ls -lh SuperGluePretrainedNetwork

In [ ]:
!cat SuperGluePretrainedNetwork/requirements.txt

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import cv2
import torch
import getpass

print("PyTorch :", torch.__version__)
print("CUDA available :", torch.cuda.is_available())
print("OpenCV  :", cv2.__version__)
print("NumPy   :", np.__version__)
print("Matplotlib :", matplotlib.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/ID26S422/The-Feature-Matching-Project.git

In [ ]:
from PIL import Image

image1_path="/content/The-Feature-Matching-Project/images/PXL_20260824_111109349.MP.jpg"
image2_path="/content/The-Feature-Matching-Project/images/PXL_20260824_111123451.jpg"
image3_path="/content/The-Feature-Matching-Project/images/PXL_20260824_111129452.jpg"

img1=np.array(Image.open(image1_path))
img2=np.array(Image.open(image2_path))
img3=np.array(Image.open(image3_path))

plt.figure(figsize=(30,10))

plt.subplot(1,3,1)
plt.imshow(img1)
plt.axis('off')
plt.title("Image 1", fontsize=20)

plt.subplot(1,3,2)
plt.imshow(img2)
plt.axis('off')
plt.title("Image 2", fontsize=20)

plt.subplot(1,3,3)
plt.imshow(img3)
plt.axis('off')
plt.title("Image 3", fontsize=20)

plt.show()

match_pairs accepts a folder with pair of images being given as text which directs it there, not images. so we create it below.

In [ ]:
import os

#assignment repository
repo_dir = "/content/The-Feature-Matching-Project"

#image filenames in panorama order
image1 = "PXL_20260824_111109349.MP.jpg"
image2 = "PXL_20260824_111123451.jpg"
image3 = "PXL_20260824_111129452.jpg"

# Create the pair file
pairs_file = os.path.join(repo_dir, "pairs.txt")

with open(pairs_file, "w") as f:
    f.write(f"{image1} {image2}\n")
    f.write(f"{image2} {image3}\n")

print("\nCreated pairs.txt:")
with open(pairs_file, "r") as f:
    print(f.read())

In [ ]:
import subprocess

#SuperGlue repository
superglue_repo = "/content/SuperGluePretrainedNetwork"

#input images and pair file from your assignment repo
input_dir = os.path.join(repo_dir, "images")
pairs_file = os.path.join(repo_dir, "pairs.txt")

#where SuperGlue results will be saved
output_dir = os.path.join(repo_dir, "results", "superglue")
os.makedirs(output_dir, exist_ok=True)

#SuperGlue matching script
match_script = os.path.join(superglue_repo, "match_pairs.py")

#running superglue match_pairs on the images and considering parameters according to input images as recommended
#by readme file in superglue repo.
command = [
    "python",
    match_script,
    "--input_dir", input_dir,
    "--input_pairs", pairs_file,
    "--output_dir", output_dir,
    "--resize", "1600",
    "--superglue", "outdoor",
    "--max_keypoints", "2048",
    "--nms_radius", "3",
    "--resize_float",
    "--viz"
]

subprocess.run(command, check=True)

print("\nResults:")
for filename in os.listdir(output_dir):
    print(filename)

In [ ]:
for file in os.listdir(output_dir):
    if file.endswith(".png"):
        img = Image.open(os.path.join(output_dir, file))

        plt.figure(figsize=(20, 10))
        plt.imshow(img)
        plt.axis("off")
        plt.title(file)
        plt.show()

In [ ]:
files = [f for f in os.listdir(output_dir) if f.endswith(".npz")]

data1 = np.load(os.path.join(output_dir, files[0]))
data2 = np.load(os.path.join(output_dir, files[1]))

In [ ]:
#finding valid matches

keypoints0_1 = data1["keypoints0"]
keypoints1_1 = data1["keypoints1"]
matches_1 = data1["matches"]
match_confidence_1 = data1["match_confidence"]
valid_matches_1 = matches_1 > -1 #meaning there actually is a match
good_matches_1 = valid_matches_1 & (match_confidence_1 > 0.75) #meaning match confidence is more than 0.75.

print("looking at matches between images 1,2")
print("Keypoints in image 0:", len(keypoints0_1))
print("Keypoints in image 1:", len(keypoints1_1))
print("Valid matches:", valid_matches_1.sum())
print("good matches:", good_matches_1.sum())

keypoints0_2 = data2["keypoints0"]
keypoints1_2 = data2["keypoints1"]
matches_2 = data2["matches"]
match_confidence_2 = data2["match_confidence"]
valid_matches_2 = matches_2 > -1
good_matches_2 = valid_matches_2 & (match_confidence_2 > 0.5)

print("looking at matches between images 2,3")
print("Keypoints in image 1:", len(keypoints0_2))
print("Keypoints in image 2:", len(keypoints1_2))
print("Valid matches:", valid_matches_2.sum())
print("good matches:", good_matches_2.sum())

In [ ]:
!git config --global user.email "id26s422@smail.iitm.ac.in"
!git config --global user.name "N P J Vedhaanth"

print("Enter your GitHub Personal Access Token:")
token = getpass.getpass()

%cd /content/The-Feature-Matching-Project

repo_url = f"https://{token}@github.com/ID26S422/The-Feature-Matching-Project.git"
os.system(f"git remote set-url origin {repo_url}")

!git pull origin main --rebase

!git add .

!git commit -m "cleanup"
!git push origin main